In [ ]:
%load_ext autoreload
%autoreload 2

import os
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from trace_comparison.data.dataset import load_trace_dataset
from trace_comparison.training.imitation import ImitationTask, ImitationModel
from trace_comparison.training.common import load_model
from utils.notebook_utils import reset_cwd_and_env
from utils.display import display_grid

reset_cwd_and_env()
base_path = Path(os.path.join(os.getcwd()))

In [ ]:
trace_dataset = load_trace_dataset(
    base_path / "data" / "traces_dataset" / 
    "nonograms" / "base_1000.jsonl")
imitation_task = ImitationTask(trace_dataset)
imitation_data = imitation_task.build_data(
    ds=trace_dataset,
    k=5,
    val_frac=0.1,
    seed=42,)

In [ ]:
checkpoint_path = base_path / "checkpoints" / "imitation" / "10.pt"
model, payload = load_model(checkpoint_path, device="cpu")

In [ ]:
print(imitation_data.train[0].shape)
print(imitation_data.train[1].shape)
print(imitation_data.train[2].shape)
print(imitation_data.train[0][0], imitation_data.train[1][0])

In [ ]:
def plot_grids(grids, predictions, clues, title=None, ncols=5):
    rows = [("Target", grids), ("Prediction", predictions)]

    nrows = len(rows)
    nitems = max(len(items) for _, items in rows)
    nplots = max(nitems, 1)
    ncols = min(ncols, nplots)

    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(3 * ncols, 3 * nrows),
        squeeze=False,
    )

    for row, (row_title, items) in enumerate(rows):
        axes[row, 0].set_ylabel(row_title, fontsize=12)

        for i, grid in enumerate(items):
            display_grid(
                axes[row, i],
                grid.detach().numpy() if hasattr(grid, "detach") else grid,
                clues=clues,
            )

        for i in range(len(items), ncols):
            axes[row, i].axis("off")

    if title:
        fig.suptitle(title, fontsize=32)

    fig.tight_layout()
    plt.show()


In [ ]:
labels = ['ac_sgl_1', 'ac_sgl_2', 'ac_sgl_3', 'ac_pop_1', 'ac_pop_2', 'ac_pop_3',
          'ac_prl_1', 'bs_pop_1', 'bs_pop_2', 'bs_pop_3', 'bs_sgl_1', 'bs_sgl_2',
          'bs_sgl_3', 'gd_adm_1', 'gd_sgd_1', 'ga_rlt_1', 'ga_rlt_2', 'ga_rlt_3',
          'ga_tmt_1', 'ga_tmt_2', 'ga_tmt_3', 'nn_rml_1']
ncols = 5

def puzzle_ids(split):
    clues = split[0]                      # (M, F)
    _, inv = np.unique(clues, axis=0, return_inverse=True)
    return inv                            # (M,) ints 0..P-1, one per row

ids = puzzle_ids(imitation_data.train)
puzzle_0_indexes = np.where(ids == 1)[0]

def display_pair(label_idx):
    puzzle = puzzle_0_indexes[label_idx]
    clues = imitation_data.train[0][puzzle] * 5
    clues = clues.reshape(2, 5, 3)

    # plot_grids(imitation_data.train[2][puzzle], clues)

    predictions = model.forward(
        imitation_data.train[0][puzzle].unsqueeze(0),
        imitation_data.train[1][puzzle].unsqueeze(0),
    )
    predictions = predictions.reshape(-1, 5, 5)

    plot_grids(
        imitation_data.train[2][puzzle],
        predictions.detach().numpy(),
        clues, title=f"{labels[label_idx]}", ncols=ncols)

In [ ]:
for label_idx in range(len(puzzle_0_indexes)):
    display_pair(label_idx)